# 第11課 - 模型上下文協議（MCP）

**模型上下文協議（MCP）** 是一個開放標準，使代理能夠在運行時動態發現並使用工具、資源和數據來源。MCP讓代理不再將工具硬編碼，而是連接到按需公開功能的外部伺服器。

在本課程中，你將學習：
- 什麼是MCP以及為什麼它對代理系統很重要
- MCP用戶端-伺服器架構的運作方式
- 如何構建使用MCP風格工具發現的代理


## 設定

**先決條件：**
- 已部署模型的 Microsoft Foundry 項目
- 執行 `az login` 進行身份驗證


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## 什麼是模型上下文協議 (MCP)？

MCP 定義了一種標準方式，讓 AI 代理能夠發現並與外部工具及數據來源互動：

- **MCP 伺服器**：通過標準協議公開工具、資源和提示
- **MCP 用戶端**：連接到伺服器並發現可用功能的代理執行時
- <strong>動態發現</strong>：代理無需硬編碼工具 — 它們在執行時發現可用資源

這對構建可擴展的代理系統非常有效，可以在不修改代理代碼的情況下新增功能。


## MCP 如何運作

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. 代理程式（MCP 用戶端）連接到 MCP 伺服器
2. 伺服器回應可用工具的清單及其結構
3. 代理程式可在推理中呼叫任何已發現的工具
4. 結果通過相同的協議回傳


## 模擬 MCP 工具發現

由於真正的 MCP 伺服器需要運行中的伺服器程序，我們將使用模擬 MCP 連接住宿服務所提供內容的 `@tool` 函數來示範此模式。

在生產環境中，這些工具會從 MCP 伺服器動態發現，而非本地定義。


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## 使用 MCP 風格工具構建代理


In [ ]:
agent = client.as_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## 生產環境中的 MCP

在生產環境中，MCP 實現了強大的模式：

- <strong>動態工具發現</strong>：代理連接到 MCP 伺服器並在運行時發現工具
- <strong>解耦架構</strong>：工具提供者可以獨立於代理進行更新
- <strong>跨組織共享</strong>：團隊可以通過 MCP 伺服器公開能力，任何代理都能使用
- **Microsoft Agent Framework 支援**：MAF 透過 `mcp` 整合內建支持 MCP 客戶端

若要在 MAF 中使用真實的 MCP 伺服器，您可透過 `hosted_mcp_tool()` 或 MCP 客戶端整合進行連接。

**了解更多：**
- [MCP 規格](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP 支援](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## 摘要

在本課程中，您學到：
- **MCP** 是一種用於代理與工具提供者之間動態工具發現的開放標準
- **客戶端-伺服器架構** 讓代理能在執行時發現功能
- MCP 支援 **可擴展、解耦的代理系統**，能無需程式碼更改添加工具
- Microsoft Agent Framework 提供生產環境使用的 **內建 MCP 支援**


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們力求準確，但請注意，自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議尋求專業人工翻譯。我們不對因使用本翻譯而引起的任何誤解或曲解承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
